# SBS Radar — preparación y recuperación local con datos reales

SK11/SK12. Reutiliza seis PDFs sellados, ejecuta comparación SK03, reconstruye un índice SK04 con 135 embeddings reales compatibles y cura tablas SK06. Verifica idempotencia y recuperación en una copia aislada. No activa red, inferencia ni recursos cloud. El horario 08:00 America/Lima sigue PAUSED. El resultado es una preparación local, no una promoción del agente.

In [ ]:
from pathlib import Path
import os, sys, tempfile, json, hashlib, shutil
from dataclasses import replace
try:
    dbutils.widgets.text("project_root", "")
    dbutils.widgets.text("state_root", "")
    dbutils.widgets.text("mode", "reuse_sealed")
    project_param=dbutils.widgets.get("project_root")
    state_param=dbutils.widgets.get("state_root")
    mode=dbutils.widgets.get("mode")
except NameError:
    project_param=os.environ.get("SBS_PROJECT_ROOT", "")
    state_param=os.environ.get("SBS_REFRESH_STATE", "")
    mode="reuse_sealed"
if project_param:
    project=Path(project_param).resolve()
else:
    candidates=[Path.cwd(),*Path.cwd().parents]
    project=next((p for p in candidates if (p/"src/sbs/operations").is_dir()),None)
if project is None or not (project/"runs/sk02-amendments-capture.json").exists():
    raise ValueError("PROJECT_WITH_SEALED_CAPTURES_REQUIRED")
if mode!="reuse_sealed":
    raise ValueError("THIS_NOTEBOOK_ONLY_AUTHORIZES_LOCAL_REUSE")
sys.path.insert(0,str(project/"src"))
from sbs.operations import RefreshRunner, load_sealed_plan
state_parent=Path(state_param).resolve() if state_param else None
if state_parent is not None and not state_parent.is_dir():raise ValueError("STATE_PARENT_DIRECTORY_REQUIRED")
state=Path(tempfile.mkdtemp(prefix="sbs-refresh-local-",dir=state_parent))
config=json.loads((project/"config/genie-pilot-002.json").read_text())
pairs={c["pair"]["pair_id"]:c["pair"] for c in config["contexts"]}
plan=replace(load_sealed_plan(project),pairs=tuple(pairs.values()))
from sbs.operations.preparers import build_real_hooks
hooks=build_real_hooks(project)
original_hashes={str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in plan.originals.values()}
assert len(original_hashes)==6
index_files=[p for p in (project/"data/retrieval/sk04-real-001").rglob("*") if p.is_file()]
index_hashes={str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in index_files}
runner=RefreshRunner(plan,state)
print({"mode":mode,"sources":len(original_hashes),"network_enabled":False,"schedule_enabled":False})

In [ ]:
baseline=runner.bootstrap_capture()
previous=runner.current()
unchanged=runner.run(run_id="notebook-sealed-reuse")
assert unchanged["status"]=="unchanged_bytes"
assert unchanged["network_requests"]==0
assert runner.run(run_id="notebook-sealed-reuse")==unchanged
assert runner.current()==previous
print({"capture_scope":baseline["release_scope"],"source_count":len(baseline["sources"]),"status":unchanged["status"],"cost":unchanged["cost"],"legal_changes":"not_assessed"})

In [ ]:
# Copia aislada; fuentes e índice entregado permanecen intactos.
recovery=Path(tempfile.mkdtemp(prefix="sbs-refresh-recovery-"))/"copied-state"
shutil.copytree(state,recovery)
recovery_runner=RefreshRunner(plan,recovery)
copy_before=recovery_runner.current()
failed=recovery_runner.run(run_id="injected-failure",hooks=hooks,force_revalidate=True,fail_at="before_publish")
assert failed["status"]=="failed" and failed["error_code"]=="INJECTED_FAILURE"
assert recovery_runner.current()==copy_before
recovered=recovery_runner.run(run_id="recovery-check",hooks=hooks,force_revalidate=True)
assert recovered["status"]=="published" and recovered["real_preparation_completed"] is True
assert recovered["production_validated"] is False
release=recovery_runner._release()
prepared={k:json.loads((recovery/v["artifact_path"]).read_text())["payload"] for k,v in release["prepared"].items()}
assert set(prepared)=={"SK03","SK04","SK06"}
assert len(prepared["SK03"]["comparisons"])==2
assert len(prepared["SK04"]["records"])==135 and prepared["SK04"]["embedding_calls"]==0
assert len(prepared["SK06"]["bundle"]["tables"]["provisions"])==135
assert runner.current()==previous
assert all(hashlib.sha256(Path(p).read_bytes()).hexdigest()==h for p,h in original_hashes.items())
assert all(hashlib.sha256(Path(p).read_bytes()).hexdigest()==h for p,h in index_hashes.items())
report={"six_pdf_reuse":True,"network_requests":0,"provider_inference_calls":0,"fault":"INJECTED_FAILURE","previous_release_preserved_on_failure":True,"recovery_status":recovered["status"],"original_and_index_hashes_preserved":True,"downstream_hooks":"real_local_preparation","compared_pairs":2,"indexed_raw_pages":135,"curated_raw_provisions":135,"structural_annotations_reused":False,"cost":None,"schedule":"PAUSED","state_root":str(state),"recovery_root":str(recovery),"runtime_promoted":False,"cloud_published":False,"e2e_agent_validated":False}
print(json.dumps(report,ensure_ascii=False,indent=2))


## Límites

Los preparadores ejecutan código real con documentos y vectores reales. Esta corrida compara páginas y cura 135 disposiciones raw; no sustituye el export 002 de 141 disposiciones ni reutiliza automáticamente sus seis anotaciones estructurales. La preparación no se publica en el runtime ni en Genie cloud. Cambios de input/modelo sin embeddings compatibles quedan pendientes; no se fabrican vectores. Reusar bytes sellados no comprueba frescura remota ni ausencia de modificaciones jurídicas. El bloqueo POSIX es local; hacen falta coordinación distribuida, release compatible, permisos y pruebas cloud antes de activar el job diario.